# SolarJEPA: reproducible GOES-16 presentation EDA

Read-only analysis of the **existing V1** May–August 2024 science-quality one-minute archive. No download, fitting, checkpoint modification or V2 holdout discovery occurs. Run with the project environment (`requirements-v1-lock.txt`); optional Jupyter execution tools are in `requirements-presentation.txt`. Install this checkout normally with `python -m pip install --no-deps .` first.

**DOCUMENTED:** XRS-A is 0.05–0.4 nm and XRS-B 0.1–0.8 nm; units are irradiance W m⁻². Native `xrsa_flux`/`xrsb_flux` already have electron correction; “raw” here means before repository QC, not detector counts. [NOAA guide](https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes16/l2/docs/GOES-R_XRS_L2_Data_Users_Guide.pdf), accessed 2026-10-10.

**OBSERVED** results below come from hash-verified files. **INFERRED** interpretations apply to this sample. Operational availability and generalization are **UNKNOWN**.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/experiments/v1_goes_2024.json").is_file())
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
from solar_flare_jepa.presentation.eda import (
    load_v1_frame, longest_complete_block, stationarity_analysis, dependence_analysis)
from solar_flare_jepa.presentation.figures import eda_figures
from solar_flare_jepa.presentation.pipeline import write_eda_evidence


## 1. Load the pinned selections and verify identities

The loader calls the existing manifest and native-contract validators. Only the four explicit V1 selections are read. It checks timestamps, units, channel status/electron flags and averaged-measurement counts. No absent or rejected values are filled.

In [2]:
frame, evidence = load_v1_frame(ROOT)
print({k: evidence[k] for k in ("source_files", "source_bytes")})
print({k: evidence["native_contract"][k] for k in
       ("observed_start_utc", "observed_end_utc", "observed_rows", "missing_bins", "duplicate_timestamps")})


{'source_files': 123, 'source_bytes': 36535474}
{'observed_start_utc': '2024-05-01T00:00:00+00:00', 'observed_end_utc': '2024-09-01T00:00:00+00:00', 'observed_rows': 177120, 'missing_bins': 0, 'duplicate_timestamps': 0}


## 2. Data quality and distributions

Missing timestamp bins, nonfinite native flux and QC-rejected measurements are different quantities. Nonfinite flux is a subset of rejected bins. V1 keeps channel-specific masks and requires complete context/target support for model evaluation. The `*_num` fields count averaged measurements, not exposure durations.

In [3]:
quality = pd.DataFrame(evidence["native_contract"]["channels"]).T
print(quality[["valid_bins", "quality_rejected_bins", "nonfinite_raw_bins", "nonpositive_raw_bins"]].to_string())
for ch in ("xrsa", "xrsb"):
    values = frame[ch + "_flux"].dropna()
    print(ch, "QC-valid irradiance quantiles (W/m2):", values.quantile([.01, .5, .95, .99]).to_dict())


     valid_bins quality_rejected_bins nonfinite_raw_bins nonpositive_raw_bins
xrsa     172646                  4474               2153                    0
xrsb     174845                  2275               2153                    0
xrsa QC-valid irradiance quantiles (W/m2): {0.01: 1.8953707670021912e-08, 0.5: 8.461841005669157e-08, 0.95: 1.2870030161593604e-06, 0.99: 8.652825454191707e-06}
xrsb QC-valid irradiance quantiles (W/m2): {0.01: 1.1771052049880383e-06, 0.5: 3.252227770644822e-06, 0.95: 1.4311658560472969e-05, 0.99: 5.1256095903226846e-05}


## 3. Flux, histogram, rolling and daily statistics

The raw figure uses native-unit linear axes. The transformed figure uses log₁₀(F / 1 W m⁻²), with gaps retained. Logarithms spread an orders-of-magnitude range; they do not guarantee stationarity. Trailing mean/variance require all 360 consecutive valid bins in each six-hour window. Daily IQR describes within-day variation, not uncertainty of an estimated median or daily seasonality.

In [4]:
train_end = "2024-07-22T00:00:00Z"
block = longest_complete_block(frame, "2024-05-01T00:00:00Z", train_end)
print("Diagnostic block:", block.index[0].isoformat(), "to",
      (block.index[-1] + pd.Timedelta(minutes=1)).isoformat(), "exclusive; rows:", len(block))
assert (np.diff(block.index.asi8) == 60_000_000_000).all()


Diagnostic block: 2024-06-11T16:38:00+00:00 to 2024-06-18T17:47:00+00:00 exclusive; rows: 10149


## 4. ACF and PACF

Diagnostics use the **earliest longest jointly valid contiguous training run**; no drop-NaN concatenation bridges gaps. ACF (FFT, biased covariance normalization) describes marginal lag dependence; PACF (Yule–Walker MLE) conditions on shorter lags. Levels and first log differences are shown separately. Correlation decay/order heuristics are not automatic model selection. No iid significance bands are asserted.

In [5]:
dependence = dependence_analysis(block)
for ch, result in dependence.items():
    print(ch, "difference ACF at selected lags:", result["selected_lags"]["log_difference"])
    print(ch, "lag-1 squared-difference ACF:", result["acf"]["squared_log_difference"][1])


xrsa difference ACF at selected lags: {'60': {'acf': -0.00985723043673743, 'pairs': 10088}, '120': {'acf': 0.007820901912994522, 'pairs': 10028}, '1440': {'acf': 0.01421248555854344, 'pairs': 8708}, '2880': {'acf': 0.009837328649996778, 'pairs': 7268}, '4320': {'acf': -0.002659816358486883, 'pairs': 5828}}
xrsa lag-1 squared-difference ACF: 0.2970048019001848
xrsb difference ACF at selected lags: {'60': {'acf': -0.01603193210672069, 'pairs': 10088}, '120': {'acf': 0.008157815796260716, 'pairs': 10028}, '1440': {'acf': 0.008161395685846265, 'pairs': 8708}, '2880': {'acf': 0.00464194108211195, 'pairs': 7268}, '4320': {'acf': -0.007595743557184349, 'pairs': 5828}}
xrsb lag-1 squared-difference ACF: 0.44871756013461933


## 5. ADF and complementary KPSS

ADF with an intercept: **H₀** is a unit root; **H₁** is stationarity around a constant under the tested autoregressive specification. Fixed 30 augmentation lags account for short-lag dependence; 10/60-lag sensitivities are retained. KPSS with an intercept: **H₀** is level stationarity; **H₁** is a nonstationary stochastic component. Automatic HAC bandwidth is recorded. KPSS's table bounds imply `<0.01` or `>0.10`, not exact endpoint p-values. These log-difference ADF p-values return zero because their statistics are below statsmodels' lower MacKinnon approximation cutoff; this is not literal probability zero or floating-point underflow.

Tests assume appropriate deterministic/lag specification and suitable dependence/moment behavior. Flares, heavy tails and regime changes can distort size. The 10,149-row block gives high power; statistical rejection is not an effect size. These are exploratory diagnostics on one training segment, not proof about all four months.

In [6]:
stationarity = stationarity_analysis(block)
view = pd.DataFrame(stationarity)[["channel", "transform", "input_rows", "adf_statistic", "adf_p_value", "adf_lags", "kpss_statistic", "kpss_p_relation", "kpss_p_value", "kpss_lags"]]
print(view.to_string(index=False))


channel      transform  input_rows  adf_statistic  adf_p_value  adf_lags  kpss_statistic kpss_p_relation  kpss_p_value  kpss_lags
   xrsa     raw_scaled       10149     -12.428812 3.982895e-23        30        0.137554               >      0.100000         55
   xrsa      log_level       10149      -8.763016 2.642788e-14        30        0.554285               =      0.029440         58
   xrsa log_difference       10148     -22.394736 0.000000e+00        30        0.002450               >      0.100000         15
   xrsb     raw_scaled       10149     -10.063968 1.311066e-17        30        0.265492               >      0.100000         57
   xrsb      log_level       10149      -8.223895 6.321841e-13        30        0.658212               =      0.017344         58
   xrsb log_difference       10148     -21.190637 0.000000e+00        30        0.002735               >      0.100000         13


## 6. Interpretation and SARIMA decision

The log-level ADF/KPSS combination can reject a unit root and reject level stationarity simultaneously. That does not prove a unit-root process or complete stationarity; finite-sample power, specification and changing solar activity matter. Differenced-log tests and short-lag dependence are useful modeling diagnostics, with the same limitations.

Squaring log increments exposes volatility persistence; changing rolling variance does not by itself establish a unit root. Seasonal-lag ACF and an exploratory periodogram are descriptive only. A single seven-day valid block cannot validate a stable daily solar period, and one-minute sampling is not evidence for daily seasonality. SARIMA is omitted from the presentation benchmark; no new fitting is needed.

In [7]:
for ch in ("xrsa", "xrsb"):
    r = {x["transform"]: x for x in stationarity if x["channel"] == ch}
    print(ch, "log level: ADF rejects at .05 =", r["log_level"]["adf_p_value"] < .05,
          "; KPSS rejects at .05 =", r["log_level"]["kpss_p_value"] < .05)
    print(ch, "log differences: ADF p =", r["log_difference"]["adf_p_value"],
          "; KPSS p", r["log_difference"]["kpss_p_relation"], r["log_difference"]["kpss_p_value"])


xrsa log level: ADF rejects at .05 = True ; KPSS rejects at .05 = True
xrsa log differences: ADF p = 0.0 ; KPSS p > 0.1
xrsb log level: ADF rejects at .05 = True ; KPSS rejects at .05 = True
xrsb log differences: ADF p = 0.0 ; KPSS p > 0.1


## 7. Generate all eight EDA figures and compact numerical evidence

PNG figures use 220 dpi; SVG variants preserve labels while rasterizing long time-series marks to limit file size. Every figure has a source/caption record in the complete CLI-generated `figure_manifest.json`. Notebook execution retains small text outputs and links rather than embedding full-resolution figures or measurements.

In [8]:
figure_entries = eda_figures(frame, block, dependence, evidence, ROOT / "docs/figures/presentation")
report = write_eda_evidence(ROOT, evidence, block, stationarity, dependence)
print("Generated", len(figure_entries), "EDA figure pairs (PNG + SVG).")
print("Numerical evidence: docs/presentation/eda_evidence.json and stationarity.csv")


Generated 8 EDA figure pairs (PNG + SVG).
Numerical evidence: docs/presentation/eda_evidence.json and stationarity.csv


[Raw flux](../docs/figures/presentation/eda_raw_flux.png) · [Log flux](../docs/figures/presentation/eda_log_flux.png) · [Distribution](../docs/figures/presentation/eda_distribution.png) · [Rolling statistics](../docs/figures/presentation/eda_rolling_statistics.png) · [ACF](../docs/figures/presentation/eda_acf.png) · [PACF](../docs/figures/presentation/eda_pacf.png) · [Daily variability](../docs/figures/presentation/eda_daily_variability.png) · [Data quality](../docs/figures/presentation/eda_data_quality.png)

## Limits and reproduction

This retrospective GOES-only sample has no independent event labels, operational delivery measurements or cycle-wide validation. Complete-case forecast scores do not represent all issue times. Descriptive full-period EDA does not tune or change V1 models. August is an already inspected historical benchmark and is not fresh confirmatory data.

The full asset command is `python scripts/build_presentation.py` from the installed project environment. [EDA report](../docs/PRESENTATION_EDA_REPORT.md), [comparison report](../docs/PRESENTATION_MODEL_COMPARISON.md), [slide guide](../docs/PRESENTATION_GUIDE.md).

Statistical references: [ADF API](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html), [KPSS API](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.kpss.html). Exact executed version: statsmodels 0.14.5; the online stable documentation can describe a newer release.